# Enviar imagem para o Gemini (SDK Google)

Este notebook usa imagens da pasta `images` e envia uma delas para o Gemini.

## Padrao de documentacao deste notebook
- Toda nova etapa deve ter **titulo** e **descricao** em celula markdown.
- Toda celula de codigo deve incluir **comentarios curtos** explicando a intencao.

Passos para executar:
1. Preencha `GEMINI_API_KEY` no arquivo `.env`.
2. No terminal, rode `uv run jupyter lab` (ou `uv run jupyter notebook`).
3. Execute as celulas em ordem.

## 1) Configuracao do ambiente e cliente Gemini

Esta etapa carrega variaveis do `.env`, valida a presenca da chave da API e inicializa o cliente da SDK do Google.

> Comentario: mantenha a chave apenas no `.env` e nunca diretamente no notebook.

In [1]:
# Importa bibliotecas para arquivo, ambiente, exibicao e SDK Gemini.
from pathlib import Path
from dotenv import load_dotenv
import mimetypes
import os

from IPython.display import Image, display
from google import genai
from google.genai import types

# Carrega variaveis do arquivo .env.
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

# Garante que a chave foi definida antes de continuar.
if not api_key:
    raise ValueError("Defina GEMINI_API_KEY no arquivo .env")

# Inicializa o cliente Gemini usando a chave da API.
client = genai.Client(api_key=api_key)
print("Cliente Gemini inicializado com sucesso.")

Cliente Gemini inicializado com sucesso.


## 2) Selecao da imagem de entrada

A celula abaixo lista os arquivos da pasta `images`, permite escolher um indice e exibe a imagem escolhida.

> Comentario: para trocar a imagem, altere apenas `selected_index`.

In [ ]:
# Define pasta de entrada e extensoes de imagem permitidas.
images_dir = Path("images")
valid_ext = {".jpg", ".jpeg", ".png", ".webp", ".gif"}

# Coleta e ordena os arquivos validos encontrados na pasta.
image_files = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in valid_ext])
if not image_files:
    raise FileNotFoundError("Nenhuma imagem encontrada em 'images'.")

# Exibe os arquivos para facilitar a escolha do indice.
for idx, img in enumerate(image_files):
    print(f"[{idx}] {img.name}")

# Troque o indice para escolher outra imagem.
selected_index = 0
image_path = image_files[selected_index]
print(f"Imagem selecionada: {image_path}")

# Mostra a imagem selecionada no notebook.
display(Image(filename=str(image_path)))

## 3) Definicao do prompt

Aqui voce escreve a instrucao que o Gemini usara para analisar a imagem.

> Comentario: prompts mais especificos costumam gerar respostas mais consistentes.

In [ ]:
# Define a instrucao textual que guiara a analise da imagem pelo modelo.
prompt = "Descreva esta imagem em detalhes e destaque os objetos principais."

## 4) Envio da imagem para o Gemini

Nesta etapa, o notebook converte a imagem em bytes, detecta o `mime_type` e envia o conteúdo para o modelo Gemini junto com o prompt.

> Comentario: altere o nome do modelo se quiser testar custo/latencia diferentes (por exemplo, `gemini-2.5-pro`).

In [ ]:
# Le o arquivo da imagem em bytes para envio a API.
image_bytes = image_path.read_bytes()

# Tenta identificar o tipo MIME com base na extensao do arquivo.
mime_type, _ = mimetypes.guess_type(str(image_path))
mime_type = mime_type or "application/octet-stream"

# Envia prompt + imagem para o Gemini e recebe a resposta textual.
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        prompt,
        types.Part.from_bytes(data=image_bytes, mime_type=mime_type),
    ],
)

# Exibe o texto retornado pelo modelo.
print(response.text)